# Importing Libraries

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Loading Preprocessed Data

In [2]:
print("Loading data...")
train_df = pd.read_csv('../data/processed/processed_training_data.csv')
test_df = pd.read_csv('../data/processed/processed_testing_data.csv')

# Separate features (X) and target (y)
# Assuming the target column was saved as 'Target' from the previous step
X_train = train_df.drop('Target', axis=1)
y_train = train_df['Target']

X_test = test_df.drop('Target', axis=1)
y_test = test_df['Target']

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Loading data...
Training features shape: (960000, 7)
Testing features shape: (500000, 7)


# Initialize the Models

In [9]:
# Storing models in a dictionary makes it easy to loop through them
models = {
    "Random Forest": RandomForestClassifier(random_state=42, max_depth= 20, min_samples_leaf= 1, min_samples_split= 2, n_estimators= 200),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, learning_rate= 0.05, max_depth= 3, n_estimators= 100, subsample= 1),
}

# Train and Evaluate Models

In [11]:
for model_name, model in models.items():
    print(f"--- Training {model_name} ---")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    
    # Print the results
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("Confusion Matrix:")
    print(conf_matrix)
    print("Classification Report:")
    print(class_report)

--- Training Random Forest ---
Accuracy: 73.12%
Confusion Matrix:
[[147321 102679]
 [ 31698 218302]]
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.59      0.69    250000
           1       0.68      0.87      0.76    250000

    accuracy                           0.73    500000
   macro avg       0.75      0.73      0.73    500000
weighted avg       0.75      0.73      0.73    500000

--- Training XGBoost ---


c:\Users\khush\OneDrive\Desktop\tennessee-eastman-anomaly-detector\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:42:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 64.67%
Confusion Matrix:
[[151198  98802]
 [ 77829 172171]]
Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.60      0.63    250000
           1       0.64      0.69      0.66    250000

    accuracy                           0.65    500000
   macro avg       0.65      0.65      0.65    500000
weighted avg       0.65      0.65      0.65    500000



# Hyperparameter Tuning

In [16]:
import time
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns

print("Setting up XGBoost hyperparameter tuning with GridSearchCV...")

# Define the base model
# tree_method='hist' is highly recommended for large datasets to speed up training
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist', 
    random_state=42
)

# Define the parameter grid to search
# Total combinations here: 2 * 2 * 3 * 2 = 24 combinations. 
# With cv=3, that is 72 total model trainings.
param_grid = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

# Configure GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='recall', # F1 is great for anomaly/fault detection
    cv=3,         # 3-fold cross-validation
    verbose=2,    # Will print out the progress
    n_jobs=-1     # Uses all available CPU cores
)

# Execute the search
start_time = time.time()
print("Starting Grid Search... this may take a while depending on your CPU.")
grid_search.fit(X_train, y_train)
end_time = time.time()

print(f"\n--- Tuning Complete! ---")
print(f"Time taken: {(end_time - start_time) / 60:.2f} minutes")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Recall Score: {grid_search.best_score_:.4f}")

# Extract the best model
best_xgb_model = grid_search.best_estimator_

Setting up XGBoost hyperparameter tuning with GridSearchCV...
Starting Grid Search... this may take a while depending on your CPU.
Fitting 3 folds for each of 24 candidates, totalling 72 fits

--- Tuning Complete! ---
Time taken: 5.11 minutes
Best Parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best Cross-Validation Recall Score: 1.0000


In [6]:
import time
from sklearn.model_selection import GridSearchCV

print("Setting up Random Forest hyperparameter tuning with GridSearchCV...")

# Define the base model
# class_weight='balanced' is critical for anomaly detection/imbalanced data
rf_base = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1 # Uses all CPU cores for the base model's trees
)

# Define the parameter grid to search
# Kept relatively small because GridSearch is exhaustive and RF can be slow.
# Total combinations: 2 * 3 * 2 * 2 = 24 combinations.
param_grid = {
    'n_estimators': [100, 200],             # Number of trees
    'max_depth': [10, 15, 20],            # Maximum depth of the trees
    'min_samples_split': [2, 5],            # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2]              # Minimum samples required to be at a leaf node
}

# Configure GridSearchCV
grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='recall', # Optimizing specifically for Recall!
    cv=3,             # 3-fold cross-validation
    verbose=2,        # Prints progress
    n_jobs=-1         # Uses all available CPU cores for the CV splits
)

# Execute the search
start_time = time.time()
print("Starting Grid Search... this may take a while depending on your CPU.")
grid_search_rf.fit(X_train, y_train)
end_time = time.time()

print(f"\n--- Tuning Complete! ---")
print(f"Time taken: {(end_time - start_time) / 60:.2f} minutes")
print(f"Best Parameters: {grid_search_rf.best_params_}")
print(f"Best Cross-Validation Recall Score: {grid_search_rf.best_score_:.4f}")

# Extract the best model
best_rf_model = grid_search_rf.best_estimator_

Setting up Random Forest hyperparameter tuning with GridSearchCV...
Starting Grid Search... this may take a while depending on your CPU.
Fitting 3 folds for each of 24 candidates, totalling 72 fits

--- Tuning Complete! ---
Time taken: 62.28 minutes
Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Cross-Validation Recall Score: 0.9962
